# Assignment 4: Your First Model

## Problem Statement
Predict students' final grades (G3) using demographic,
family background, and study-related features.

## Problem Type
Regression

## Dataset
UCI Student Performance Dataset (Portuguese students).

## Target
G3 - Final grade

## Models
1. Linear Regression
2. Unlimited Decision Tree Regressor
3. Dummy Regressor (Mean Baseline)

In [2]:
!wget -q https://archive.ics.uci.edu/static/public/320/student+performance.zip -O student.zip
!unzip -o student.zip
!mkdir -p student_data
!unzip -o student.zip -d student_data

Archive:  student.zip
 extracting: .student.zip_old        
 extracting: student.zip             
Archive:  student.zip
  inflating: student_data/student-mat.csv  
  inflating: student_data/student-por.csv  
  inflating: student_data/student-merge.R  
  inflating: student_data/student.txt  


In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("student_data/student-por.csv", sep=";")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (649, 33)


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,4,0,11,11
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,2,9,11,11
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,6,12,13,12
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,0,14,14,14
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,0,11,13,13


In [4]:
# Target variable
y = df["G3"]

# Exclude target and previous grades
X = df.drop(columns=["G3", "G1", "G2"])

print("Features:", X.columns.tolist())
print("Target:", "G3")
print("Feature shape:", X.shape)

Features: ['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences']
Target: G3
Feature shape: (649, 30)


In [5]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numerical features:", numeric_features)
print("Categorical features:", categorical_features)

Numerical features: ['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences']
Categorical features: ['school', 'sex', 'address', 'famsize', 'Pstatus', 'Mjob', 'Fjob', 'reason', 'guardian', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 519
Testing rows: 130


In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

model.fit(X_train, y_train)

train_score = model.score(X_train, y_train)
test_score = model.score(X_test, y_test)

print("Linear Regression")
print("Training R2:", train_score)
print("Testing R2:", test_score)

Linear Regression
Training R2: 0.38744907311128973
Testing R2: 0.1601699196431835


In [8]:
gap = train_score - test_score

print("Training score:", train_score)
print("Testing score:", test_score)
print("Score gap:", gap)

if gap > 0.15:
    print("There may be overfitting.")
else:
    print("The train-test score gap is relatively small.")

Training score: 0.38744907311128973
Testing score: 0.1601699196431835
Score gap: 0.22727915346810623
There may be overfitting.


In [9]:
greedy = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", DecisionTreeRegressor(random_state=42))
])

greedy.fit(X_train, y_train)

greedy_train = greedy.score(X_train, y_train)
greedy_test = greedy.score(X_test, y_test)

print("Unlimited Decision Tree")
print("Training R2:", greedy_train)
print("Testing R2:", greedy_test)

Unlimited Decision Tree
Training R2: 1.0
Testing R2: -0.4806072668139123


In [10]:
dummy = DummyRegressor(strategy="mean")
dummy.fit(X_train, y_train)

baseline_score = dummy.score(X_test, y_test)

print("Mean Baseline R2:", baseline_score)
print("Linear Regression R2:", test_score)

if test_score > baseline_score:
    print("Linear Regression beats the baseline.")
else:
    print("Linear Regression does not beat the baseline.")

Mean Baseline R2: -0.032159804579835916
Linear Regression R2: 0.1601699196431835
Linear Regression beats the baseline.


In [11]:
comparison = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Unlimited Decision Tree",
        "Mean Baseline"
    ],
    "Training R2": [
        train_score,
        greedy_train,
        dummy.score(X_train, y_train)
    ],
    "Testing R2": [
        test_score,
        greedy_test,
        baseline_score
    ]
})

comparison

,Model,Training R2,Testing R2
0,Linear Regression,0.387449,0.160170
1,Unlimited Decision Tree,1.000000,-0.480607
2,Mean Baseline,0.000000,-0.032160


## Model Evaluation and Findings

### 1. Linear Regression Performance

The Linear Regression model achieved a training R² score of 0.3874 and a testing R² score of 0.1602. This indicates that the model explains approximately 16% of the variation in final grades on unseen data. The difference between training and testing scores suggests some overfitting.

### 2. Understanding Overfitting

The unlimited Decision Tree achieved a training R² score of 1.0 but a testing R² score of -0.4809. This shows that the model memorized the training data instead of learning patterns that generalize to unseen students. Its poor testing performance demonstrates overfitting.

### 3. Baseline Comparison

The Mean Baseline achieved a testing R² score of -0.0322, while Linear Regression achieved 0.1602. Therefore, Linear Regression performs better than the baseline model and learns some useful patterns from the dataset.

### 4. Conclusion

Linear Regression is a better starting point than the unlimited Decision Tree for this prediction task. However, its testing score is relatively low, indicating that the selected features alone do not predict final grades particularly well.

The experiment demonstrates why evaluating a model on unseen test data is important and why a simple baseline should always be used for comparison.